In [11]:
# %%
import os
import pandas as pd
import json
import subprocess
from datetime import datetime
import re
import folium
from geopy.distance import geodesic

In [ ]:
combined_df = pd.read_parquet(os.path.join("./output", "combined_data.parquet"))

In [ ]:
# %% get all locations in 500m radius from the given location
 # Berlin coordinates

In [28]:
given_location = (6.1394815, 80.0998373)


# Function to calculate distance
def is_within_radius(row, center, radius=50):
    location = (row["latitude"], row["longitude"])
    return geodesic(center, location).meters <= radius


# Filter locations within 500m radius
df_filtered_locations = combined_df[
    combined_df.apply(is_within_radius, axis=1, center=given_location)
]

In [29]:
df_filtered_locations.head()

,latitude,longitude,address,old_latitude,old_longitude,"lat,long",localTime,batteryLevel,googleAccount,lastUpdate,serverTime
3095,6.139584,80.099645,"290-250 Colombo Main Rd, Hikkaduwa, Sri Lanka",6.1542809,80.0910765,"6.1395838, 80.0996446",11:10 am,25%,Navoda Madubashini,A moment ago,"27/03/2025, 16:40:34"
3096,6.139503,80.099836,"43QX+RXJ, Amarasena Mawatha, Hikkaduwa, Sri Lanka",6.1542809,80.0910765,"6.1395027, 80.0998361",11:08 am,25%,Navoda Madubashini,A moment ago,"27/03/2025, 16:38:35"
3097,6.139492,80.099839,"43QX+RXJ, Amarasena Mawatha, Hikkaduwa, Sri Lanka",6.1542809,80.0910765,"6.1394915, 80.0998389",11:06 am,26%,Navoda Madubashini,2 min ago,"27/03/2025, 16:36:34"
3098,6.139492,80.099839,"43QX+RXJ, Amarasena Mawatha, Hikkaduwa, Sri Lanka",6.1542809,80.0910765,"6.1394915, 80.0998389",11:04 am,26%,Navoda Madubashini,A moment ago,"27/03/2025, 16:34:34"
3099,6.139486,80.099839,"43QX+RXJ, Amarasena Mawatha, Hikkaduwa, Sri Lanka",6.1542809,80.0910765,"6.1394862, 80.0998385",11:01 am,27%,Navoda Madubashini,A moment ago,"27/03/2025, 16:32:34"


In [35]:
df_filtered_locations.sort_values(by=["serverTime"], ascending=False, inplace=True)
df_filtered_locations.reset_index(drop=True, inplace=True)
df_filtered_locations["serverTime"] = pd.to_datetime(
    df_filtered_locations["serverTime"], format="%d/%m/%Y, %H:%M:%S", utc=True
)

C:\Users\mbtlf\AppData\Local\Temp\ipykernel_33076\2196585300.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\mbtlf\AppData\Local\Temp\ipykernel_33076\2196585300.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
# Show df_filtered_location on a folim map. Include all the columns clickable on the map, including the index. When it is houred the index and servertime should show.
#  When it is clicked all columns should show. There should be a link to open the location in google maps. The map should be saved as a html file.

,latitude,longitude,address,old_latitude,old_longitude,"lat,long",localTime,batteryLevel,googleAccount,lastUpdate,serverTime
0,6.159193,80.093551,"Unnamed Road, Sri Lanka",6.1567431,80.0842978,"6.1591934, 80.0935513",6:29 pm,77%,Navoda Madubashini,A moment ago,"31/03/2025, 23:59:47"
1,6.158868,80.093398,"11-67 Seenigama - Aluthwala Rd, Sri Lanka",6.1567431,80.0842978,"6.1588683, 80.0933983",6:27 pm,75%,Navoda Madubashini,1 min ago,"31/03/2025, 23:57:47"
2,6.158868,80.093398,"535V+F9W, ගාල්ල, Sri Lanka",6.1567431,80.0842978,"6.158868, 80.0933981",6:25 pm,75%,Navoda Madubashini,A moment ago,"31/03/2025, 23:55:47"
3,6.158692,80.093272,"11-67 Seenigama - Aluthwala Rd, Sri Lanka",6.1567431,80.0842978,"6.1586917, 80.0932717",6:23 pm,74%,Navoda Madubashini,A moment ago,"31/03/2025, 23:53:47"
4,6.158753,80.093502,"11-67 Seenigama - Aluthwala Rd, Sri Lanka",6.1567431,80.0842978,"6.1587533, 80.0935017",6:21 pm,73%,Navoda Madubashini,A moment ago,"31/03/2025, 23:51:47"


In [17]:
# Create a Folium map centered at the given location
map_center = given_location
folium_map = folium.Map(location=map_center, zoom_start=15)

# Add markers for each location in df_filtered_locations

In [31]:
for idx, row in df_filtered_locations.iterrows():
    popup_content = f""" <b>Index:</b> {idx}<br> <b>Latitude:</b> {row['latitude']}<br> <b>Longitude:</b>
    {row['longitude']}<br> <b>Address:</b> {row['address']}<br> <b>Old Latitude:</b> {row['old_latitude']}<br> <b>Old
        Longitude:</b> {row['old_longitude']}<br> <b>Lat, Long:</b> {row['lat,long']}<br> <b>Local Time:</b>
    {row['localTime']}<br> <b>Battery Level:</b> {row['batteryLevel']}<br> <b>Google Account:</b> {row['googleAccount']}<br>
    <b>Last Update:</b> {row['lastUpdate']}<br> <b>Server Time:</b> {row['serverTime']}<br> <a
        href="https://www.google.com/maps?q={row['latitude']},{row['longitude']}" target="_blank">Open in Google Maps</a>
    """

    popup = folium.Popup(popup_content, max_width=300)

    # Create a tooltip with index and server time
    tooltip = f"Index: {idx}, Server Time: {row['serverTime']}"

    # Add the marker to the map
    folium.Marker(
    location=(row['latitude'], row['longitude']),
    popup=popup,
    tooltip=tooltip
    ).add_to(folium_map)

    # Save the map as an HTML file
    folium_map.save("./output/filtered_locations_map.html")

In [36]:
import dtale
d =  dtale.show(df_filtered_locations)
d.open_browser()

c:\Users\mbtlf\anaconda3\Lib\site-packages\dtale\utils.py:911: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<DatetimeArray>
['2025-03-27 16:40:34', '2025-03-27 16:38:35', '2025-03-27 16:36:34',
 '2025-03-27 16:34:34', '2025-03-27 16:32:34', '2025-03-27 16:30:34',
 '2025-03-27 16:28:34', '2025-03-27 16:26:34', '2025-03-27 16:24:34',
 '2025-03-27 16:22:34',
 ...
 '2025-03-05 19:13:35', '2025-03-05 19:11:31', '2025-03-05 19:09:31',
 '2025-03-05 19:07:31', '2025-03-05 19:05:31', '2025-03-05 19:03:32',
 '2025-03-05 19:01:31', '2025-03-05 18:59:31', '2025-03-05 18:57:31',
 '2025-03-05 18:55:31']
Length: 125, dtype: datetime64[ns]' has dtype incompatible with datetime64[ns, UTC], please explicitly cast to a compatible dtype first.

